# Notebook 04 — EDA and Research Scope
**Khảo sát dữ liệu khám phá và xác định phạm vi nghiên cứu**

## 1. Mục tiêu

Thực hiện phân tích khám phá dữ liệu (Exploratory Data Analysis — EDA) nhằm khảo sát đặc điểm, phân bố, chất lượng và khả năng khai thác của bộ dữ liệu CBIS-DDSM.

Kết quả phân tích được sử dụng làm căn cứ để xác định phạm vi nghiên cứu phù hợp, bao gồm Mass, Calcification hoặc kết hợp cả hai, đồng thời đánh giá khả năng sử dụng ROI Mask và Bounding Box cho bài toán phát hiện, định vị tổn thương.

## 2. Dữ liệu đầu vào

Kế thừa kết quả từ các Notebook trước:

- Notebook 01 — Data Inventory.
- Notebook 02 — Data Quality and Linkage.
- Notebook 03 — Lesion Dataset Construction.

Dữ liệu đã xử lý được lưu tại:

`C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset`

Dữ liệu DICOM gốc được bảo toàn trong thư mục `Data\rawdata`.

## 3. Nội dung thực hiện

- Khảo sát quy mô và phân bố Mass, Calcification.
- Phân tích phân bố pathology và các thuộc tính mô tả tổn thương.
- Khảo sát số lượng bệnh nhân, annotation, candidate và ảnh.
- Phân tích bên vú và góc chụp CC/MLO.
- Khảo sát kích thước ảnh DICOM và ROI Mask.
- Phân tích diện tích, hình dạng và Bounding Box của ROI.
- Trực quan hóa Full Image, Cropped Image và ROI Mask.
- Đánh giá khả năng khai thác dữ liệu cho bài toán phát hiện và định vị.
- So sánh tính khả thi của các phạm vi nghiên cứu Mass, Calcification và kết hợp cả hai.

## 4. Nguyên tắc thực hiện

- Chỉ sử dụng dataset CBIS-DDSM.
- Không tự suy diễn hoặc tạo số liệu chưa được thống kê.
- Phân biệt rõ patient, annotation, candidate và image.
- Bảo toàn các nhãn pathology gốc.
- Không tự ý resize ROI Mask hoặc giả định Bounding Box đã cùng hệ tọa độ với Full Image.
- Kiểm soát patient-level data leakage.
- Chưa lựa chọn kiến trúc mô hình khi chưa hoàn thành khảo sát dữ liệu.

## 5. Kết quả đầu ra

- Các bảng thống kê EDA.
- Các biểu đồ phân tích phân bố dữ liệu.
- Hình ảnh minh họa Full Image, Cropped Image và ROI Mask.
- Báo cáo chất lượng và khả năng sử dụng annotation cho localization.
- Đánh giá ưu, nhược điểm của từng phạm vi nghiên cứu.
- Đề xuất phạm vi nghiên cứu chính thức cho các giai đoạn tiếp theo.

## 6. Phạm vi chưa thực hiện

Notebook này chưa triển khai xây dựng pipeline huấn luyện, lựa chọn kiến trúc học sâu cụ thể hoặc huấn luyện mô hình.

### 04.1 — Kiểm tra và nạp dữ liệu đầu vào

**Mục tiêu:** Nạp lại các bảng dữ liệu đã được xây dựng và kiểm định ở Notebook 03, đảm bảo dữ liệu sẵn sàng cho quá trình EDA.

**Đầu vào:** Các file CSV và báo cáo trong thư mục `nb03_lesion_dataset`.

In [ ]:
# 4.1.1. Thiết lập đường dẫn và nạp dữ liệu

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")
INPUT_DIR = PROJECT_ROOT / "Data" / "processed" / "nb03_lesion_dataset"

FILES = {
    "lesion_master": "lesion_master.csv",
    "review_groups": "lesion_review_groups.csv",
    "annotation_mapping": "lesion_annotation_mapping.csv",
    "image_mapping": "lesion_image_mapping.csv",
    "image_catalog": "image_path_catalog.csv",
    "roi_mapping": "lesion_roi_mapping.csv",
    "integrity_check": "nb03_integrity_check.csv",
    "summary_report": "nb03_summary_report.txt",
}

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Không tìm thấy dự án: {PROJECT_ROOT}")

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy dữ liệu: {INPUT_DIR}")

missing_files = [
    filename
    for filename in FILES.values()
    if not (INPUT_DIR / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(f"Các file bị thiếu: {missing_files}")

print(f"[ĐẠT] Tìm thấy đầy đủ {len(FILES)} file đầu vào.")
print(f"Thư mục dữ liệu: {INPUT_DIR}")

nb04_tables = {
    name: pd.read_csv(
        INPUT_DIR / filename,
        encoding="utf-8-sig",
        low_memory=False,
    )
    for name, filename in FILES.items()
    if filename.endswith(".csv")
}

nb04_summary_report = (
    INPUT_DIR / FILES["summary_report"]
).read_text(encoding="utf-8-sig")

# Gán DataFrame cho các biến sử dụng trong Notebook 04

nb04_lesion_master_df = nb04_tables["lesion_master"]
nb04_lesion_review_groups_df = nb04_tables["review_groups"]
nb04_lesion_annotation_mapping_df = nb04_tables["annotation_mapping"]
nb04_lesion_image_mapping_df = nb04_tables["image_mapping"]
nb04_image_path_catalog_df = nb04_tables["image_catalog"]
nb04_lesion_roi_mapping_df = nb04_tables["roi_mapping"]
nb04_integrity_check_df = nb04_tables["integrity_check"]

print(f"[ĐẠT] Đã nạp {len(nb04_tables)} bảng CSV và 1 báo cáo TXT.")

[ĐẠT] Tìm thấy đầy đủ 8 file đầu vào.
Thư mục dữ liệu: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset
[ĐẠT] Đã nạp 7 bảng CSV và 1 báo cáo TXT.


In [ ]:
# 4.1.2. Kiểm tra quy mô và kích thước bảng
dataframe_objects = {
    "lesion_master_df": nb04_lesion_master_df,
    "lesion_review_groups_df": nb04_lesion_review_groups_df,
    "lesion_annotation_mapping_df": nb04_lesion_annotation_mapping_df,
    "lesion_image_mapping_df": nb04_lesion_image_mapping_df,
    "image_path_catalog_df": nb04_image_path_catalog_df,
    "lesion_roi_mapping_df": nb04_lesion_roi_mapping_df,
    "nb03_integrity_check_df": nb04_integrity_check_df,
}

expected_shapes = {
    "lesion_master_df": (1989, 42),
    "lesion_review_groups_df": (140, 36),
    "lesion_annotation_mapping_df": (3568, 51),
    "lesion_image_mapping_df": (7136, 17),
    "image_path_catalog_df": (6671, 4),
    "lesion_roi_mapping_df": (3568, 87),
    "nb03_integrity_check_df": (54, 5),
}

shape_rows = []

for name, df in dataframe_objects.items():

    expected = expected_shapes[name]
    actual = df.shape

    shape_rows.append({
        "dataframe": name,
        "so_dong": actual[0],
        "so_cot": actual[1],
        "kich_thuoc_ky_vong": str(expected),
        "ket_qua": "ĐẠT" if actual == expected else "KHÔNG ĐẠT",
        "dung_luong_MB": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 3
        ),
    })

nb04_shape_check_df = pd.DataFrame(shape_rows)

display(nb04_shape_check_df)

,dataframe,so_dong,so_cot,kich_thuoc_ky_vong,ket_qua,dung_luong_MB
0,lesion_master_df,1989,42,"(1989, 42)",ĐẠT,3.915
1,lesion_review_groups_df,140,36,"(140, 36)",ĐẠT,0.262
2,lesion_annotation_mapping_df,3568,51,"(3568, 51)",ĐẠT,11.964
3,lesion_image_mapping_df,7136,17,"(7136, 17)",ĐẠT,9.021
4,image_path_catalog_df,6671,4,"(6671, 4)",ĐẠT,3.795
5,lesion_roi_mapping_df,3568,87,"(3568, 87)",ĐẠT,17.980
6,nb03_integrity_check_df,54,5,"(54, 5)",ĐẠT,0.019


In [ ]:
# 4.1.3. Kiểm tra khóa định danh

key_definitions = {
    "lesion_master_df": ["lesion_candidate_id"],
    "lesion_review_groups_df": ["review_group_id"],
    "lesion_annotation_mapping_df": ["annotation_id"],
    "lesion_image_mapping_df": ["annotation_id", "image_role"],
    "image_path_catalog_df": ["image_role", "image_path_key"],
    "lesion_roi_mapping_df": ["annotation_id"],
}

key_rows = []

for table_name, columns in key_definitions.items():

    df = dataframe_objects[table_name]

    key_data = (
        df[columns]
        .astype("string")
        .apply(lambda col: col.str.strip())
        .replace("", pd.NA)
    )

    missing_count = int(key_data.isna().any(axis=1).sum())

    valid_data = key_data.dropna()

    duplicate_count = int(
        valid_data.duplicated().sum()
    )

    key_rows.append({
        "dataframe": table_name,
        "khoa_dinh_danh": " + ".join(columns),
        "gia_tri_thieu": missing_count,
        "khoa_trung": duplicate_count,
        "ket_qua": (
            "ĐẠT"
            if missing_count == 0 and duplicate_count == 0
            else "KHÔNG ĐẠT"
        ),
    })

nb04_key_check_df = pd.DataFrame(key_rows)

display(nb04_key_check_df)

,dataframe,khoa_dinh_danh,gia_tri_thieu,khoa_trung,ket_qua
0,lesion_master_df,lesion_candidate_id,0,0,ĐẠT
1,lesion_review_groups_df,review_group_id,0,0,ĐẠT
2,lesion_annotation_mapping_df,annotation_id,0,0,ĐẠT
3,lesion_image_mapping_df,annotation_id + image_role,0,0,ĐẠT
4,image_path_catalog_df,image_role + image_path_key,0,0,ĐẠT
5,lesion_roi_mapping_df,annotation_id,0,0,ĐẠT


In [ ]:
# 4.1.4. Tổng hợp kết quả và kết luận

shape_passed = nb04_shape_check_df["ket_qua"].eq("ĐẠT").all()
key_passed = nb04_key_check_df["ket_qua"].eq("ĐẠT").all()

qa_counts = (
    nb04_integrity_check_df["ket_qua"]
    .value_counts()
    .to_dict()
)

nb03_qa_passed = qa_counts == {"ĐẠT": 54}
report_passed = bool(nb04_summary_report.strip())
nb04_input_ready = (
    shape_passed
    and key_passed
    and nb03_qa_passed
    and report_passed
)

print("=" * 65)
print("KẾT LUẬN PHẦN 04.1")
print("=" * 65)

print(f"Kích thước các bảng: {'ĐẠT' if shape_passed else 'CHƯA ĐẠT'}")
print(f"Khóa định danh: {'ĐẠT' if key_passed else 'CHƯA ĐẠT'}")
print(f"Kiểm định Notebook 03: {'ĐẠT' if nb03_qa_passed else 'CHƯA ĐẠT'}")
print(f"Báo cáo đầu vào: {'ĐẠT' if report_passed else 'CHƯA ĐẠT'}")

if not nb04_input_ready:
    raise ValueError("Dữ liệu đầu vào chưa đáp ứng điều kiện EDA.")

print("\n[ĐẠT] Dữ liệu đã sẵn sàng cho Notebook 04.")
print(f"Số Candidate: {len(nb04_lesion_master_df):,}")
print(f"Số Review Group: {len(nb04_lesion_review_groups_df):,}")
print(f"Số Annotation: {len(nb04_lesion_annotation_mapping_df):,}")
print(f"Số ROI Mapping: {len(nb04_lesion_roi_mapping_df):,}")


KẾT LUẬN PHẦN 04.1
Kích thước các bảng: ĐẠT
Khóa định danh: ĐẠT
Kiểm định Notebook 03: ĐẠT
Báo cáo đầu vào: ĐẠT

[ĐẠT] Dữ liệu đã sẵn sàng cho Notebook 04.
Số Candidate: 1,989
Số Review Group: 140
Số Annotation: 3,568
Số ROI Mapping: 3,568


### 04.2 — Khảo sát quy mô và thành phần dữ liệu

**Mục tiêu:** Khảo sát quy mô và thành phần của bộ dữ liệu CBIS-DDSM sau khi đã xây dựng các bảng liên kết ở Notebook 03.

**Nội dung thực hiện:**
- Thống kê tổng số bệnh nhân, annotation, candidate và review group.
- Phân tích phân bố Mass và Calcification.
- Thống kê số candidate và review group theo loại tổn thương.
- Khảo sát số lượng annotation theo bên vú và góc chụp.
- Thống kê ảnh Full Image, Cropped Image và số đường dẫn ảnh duy nhất.
- Đối chiếu các số liệu giữa Annotation Mapping, Image Mapping và Image Catalog.

**Nguyên tắc:** Phân biệt số bệnh nhân, annotation, candidate và đường dẫn ảnh. Các đơn vị này không được xem là tương đương.

In [16]:
# 4.2.1. Chuẩn bị dữ liệu khảo sát

# 1. KIỂM TRA DỮ LIỆU ĐẦU VÀO

if not nb04_input_ready:
    raise ValueError("Dữ liệu đầu vào chưa sẵn sàng.")

annotation_eda_df = nb04_lesion_annotation_mapping_df.copy()

required_columns = [
    "annotation_id",
    "patient_id",
    "lesion_type",
    "lesion_candidate_id",
    "review_group_id",
]

missing = [
    col for col in required_columns
    if col not in annotation_eda_df.columns
]

if missing:
    raise ValueError(f"Annotation Mapping thiếu cột: {missing}")


# 2. CHUẨN HÓA GIÁ TRỊ DÙNG CHO EDA

annotation_eda_df["_lesion_type"] = (
    annotation_eda_df["lesion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print("[ĐẠT] Dữ liệu khảo sát đã sẵn sàng.")
print(f"Số annotation: {len(annotation_eda_df):,}")

# 3. THỐNG KÊ QUY MÔ DỮ LIỆU

nb04_dataset_scale_df = pd.DataFrame([
    {
        "don_vi": "Patient",
        "so_luong": annotation_eda_df["patient_id"].nunique(),
    },
    {
        "don_vi": "Annotation",
        "so_luong": annotation_eda_df["annotation_id"].nunique(),
    },
    {
        "don_vi": "Candidate",
        "so_luong": nb04_lesion_master_df["lesion_candidate_id"].nunique(),
    },
    {
        "don_vi": "Review Group",
        "so_luong": nb04_lesion_review_groups_df["review_group_id"].nunique(),
    },
    {
        "don_vi": "Image Mapping",
        "so_luong": len(nb04_lesion_image_mapping_df),
    },
    {
        "don_vi": "ROI Mapping",
        "so_luong": len(nb04_lesion_roi_mapping_df),
    },
    {
        "don_vi": "Image Path Catalog",
        "so_luong": len(nb04_image_path_catalog_df),
    },
])

display(nb04_dataset_scale_df)

# 4. PHÂN BỐ THEO LOẠI TỔN THƯƠNG

annotation_type_summary = (
    annotation_eda_df
    .groupby("_lesion_type", dropna=False)
    .agg(
        so_annotation=("annotation_id", "nunique"),
        so_patient=("patient_id", "nunique"),
    )
    .reset_index()
    .rename(columns={"_lesion_type": "lesion_type"})
)

candidate_eda_df = annotation_eda_df.loc[
    annotation_eda_df["lesion_candidate_id"].notna()
].copy()

candidate_type_summary = (
    candidate_eda_df
    .groupby("_lesion_type", dropna=False)
    .agg(
        so_candidate=("lesion_candidate_id", "nunique"),
        so_patient=("patient_id", "nunique"),
    )
    .reset_index()
    .rename(columns={"_lesion_type": "lesion_type"})
)

review_eda_df = annotation_eda_df.loc[
    annotation_eda_df["review_group_id"].notna()
].copy()

review_type_summary = (
    review_eda_df
    .groupby("_lesion_type", dropna=False)
    .agg(
        so_review_group=("review_group_id", "nunique"),
    )
    .reset_index()
    .rename(columns={"_lesion_type": "lesion_type"})
)

nb04_lesion_type_summary_df = (
    annotation_type_summary
    .merge(candidate_type_summary, on="lesion_type", how="outer")
    .merge(review_type_summary, on="lesion_type", how="outer")
)

display(nb04_lesion_type_summary_df)

# 5. KIỂM TRA LOẠI TỔN THƯƠNG TRONG CANDIDATE

candidate_type_consistency_df = (
    candidate_eda_df
    .groupby("lesion_candidate_id")
    .agg(
        so_loai_ton_thuong=("_lesion_type", "nunique"),
        cac_loai_ton_thuong=(
            "_lesion_type",
            lambda x: ", ".join(sorted(x.dropna().unique())),
        ),
    )
    .reset_index()
)

candidate_type_conflicts_df = (
    candidate_type_consistency_df.loc[
        candidate_type_consistency_df["so_loai_ton_thuong"] > 1
    ]
    .copy()
)

print("Số Candidate:", len(candidate_type_consistency_df))
print("Candidate có nhiều loại tổn thương:", len(candidate_type_conflicts_df))

display(candidate_type_conflicts_df.head(20))

[ĐẠT] Dữ liệu khảo sát đã sẵn sàng.
Số annotation: 3,568


,don_vi,so_luong
0,Patient,1566
1,Annotation,3568
2,Candidate,1989
3,Review Group,140
4,Image Mapping,7136
5,ROI Mapping,3568
6,Image Path Catalog,6671


,lesion_type,so_annotation,so_patient_x,so_candidate,so_patient_y,so_review_group
0,calcification,1872,753,837,704,108
1,mass,1696,892,1152,874,32


Số Candidate: 1989
Candidate có nhiều loại tổn thương: 0


,lesion_candidate_id,so_loai_ton_thuong,cac_loai_ton_thuong


### Lưu ý: 
Tổng số bệnh nhân ở đây là số patient_id duy nhất trong Annotation Mapping. Một bệnh nhân có thể xuất hiện ở cả Mass và Calcification.

In [19]:
 # 6. PHÂN BỐ BÊN VÚ VÀ GÓC CHỤP

def summarize_annotation_category(df, column):

    temp = df.copy()

    temp[column] = (
        temp[column]
        .astype("string")
        .str.strip()
        .fillna("MISSING")
    )

    return (
        temp.groupby(column)
        .agg(
            so_annotation=("annotation_id", "nunique"),
            so_patient=("patient_id", "nunique"),
        )
        .reset_index()
        .sort_values("so_annotation", ascending=False)
    )


for column in ["laterality", "image_view"]:

    if column in annotation_eda_df.columns:

        print(f"\nPhân bố {column}:")

        display(
            summarize_annotation_category(
                annotation_eda_df,
                column,
            )
        )

    else:
        print(f"[THIẾU] Không tìm thấy cột {column}.")

# 7. PHÂN BỐ THEO NGUỒN DỮ LIỆU

for column in ["source_split", "source_table"]:

    if column in annotation_eda_df.columns:

        print(f"\nPhân bố {column}:")

        display(
            summarize_annotation_category(
                annotation_eda_df,
                column,
            )
        )

    else:
        print(f"[THÔNG TIN] Không có cột {column} trong dữ liệu.")

# 8. THỐNG KÊ CÁC VAI TRÒ ẢNH

image_mapping_summary = (
    nb04_lesion_image_mapping_df
    .groupby("image_role")
    .agg(
        so_quan_he=("annotation_id", "size"),
        so_annotation=("annotation_id", "nunique"),
        so_duong_dan=("image_path_key", "nunique"),
    )
    .reset_index()
)

catalog_summary = (
    nb04_image_path_catalog_df
    .groupby("image_role")
    .agg(
        so_path_trong_catalog=("image_path_key", "nunique"),
    )
    .reset_index()
)

nb04_image_summary_df = image_mapping_summary.merge(
    catalog_summary,
    on="image_role",
    how="outer",
)

display(nb04_image_summary_df)


Phân bố laterality:


,laterality,so_annotation,so_patient
0,LEFT,1819,874
1,RIGHT,1749,834



Phân bố image_view:


,image_view,so_annotation,so_patient
1,MLO,1896,1473
0,CC,1672,1312



Phân bố source_split:


,source_split,so_annotation,so_patient
1,train,2864,1248
0,test,704,349



Phân bố source_table:


,source_table,so_annotation,so_patient
1,calc_train,1546,602
3,mass_train,1318,691
2,mass_test,378,201
0,calc_test,326,151


,image_role,so_quan_he,so_annotation,so_duong_dan,so_path_trong_catalog
0,cropped_image,3568,3568,3568,3568
1,full_image,3568,3568,3103,3103


In [ ]:
# 9. KẾT LUẬN KHẢO SÁT QUY MÔ DỮ LIỆU

print("=" * 65)
print("KẾT LUẬN PHẦN 04.2")
print("=" * 65)

print(f"Tổng số bệnh nhân: {annotation_eda_df['patient_id'].nunique():,}")
print(f"Tổng số annotation: {annotation_eda_df['annotation_id'].nunique():,}")
print(f"Tổng số candidate: {len(nb04_lesion_master_df):,}")
print(f"Tổng số review group: {len(nb04_lesion_review_groups_df):,}")

print("\nPhân bố loại tổn thương:")
display(nb04_lesion_type_summary_df)

print("\nPhân bố vai trò ảnh:")
display(nb04_image_summary_df)

print("\nCandidate có loại tổn thương không nhất quán:",
      len(candidate_type_conflicts_df))